In [2]:
import requests
from bs4 import BeautifulSoup

res = requests.get('https://registrar.ucdavis.edu/calendar/archive/academic-calendar')

In [ ]:
soup = BeautifulSoup(res.content, 'html.parser')

# Use this to determine which HTML element is the target to scrape. In this case, everything is stored in 'tr' inside a 'div'
# print(soup.prettify())

finalDates = []
lines = []
# Find the main content container
content = soup.find('div')
if content:
    for para in content.find_all('tr'):
        lines.append(para.text.strip())
        if 'Finals Begin' in para.text:
            finalDates.append(para.text.strip())
else:
    print("No article content found.")

finalDates
cleanDates = []

# Removing the first 16 characters from each date string to get just the date information
for date in finalDates:
    cleanDates.append(date[16:])

# Initial list of all final dates WITHOUT cleaning
# cleanDates

['Mon-Fri12/9-13',
 'Mon-Fri3/17-21',
 'Fri, Mon-Thu6/6, 9-12',
 'Mon-Fri12/11-15',
 'Mon-Fri3/18-22',
 'Fri, Mon-Thu6/7, 10-13',
 'Mon-Fri12/5-9',
 'Mon-Fri3/20-24',
 'Fri, Mon-Thu6/9, 12-15',
 'Mon-Fri12/6-10',
 'Mon-Fri3/14-18',
 'Fri, Mon-Thu6/3, 6-9',
 'Mon-Fri12/14-18',
 'Mon-Fri3/15-19',
 'Fri, Mon-Thu6/4, 7-10',
 'Mon-Fri12/9-13',
 'Mon-Fri3/16-20',
 'Fri, Mon-Thu6/5, 8-11',
 'Mon-Fri12/10-14',
 'Mon-Fri3/18-22',
 'Fri, Mon-Thu6/7, 10-13',
 'Mon-Fri12/11-15',
 'Mon-Fri3/19-23',
 'Fri, Mon-Thu6/8, 11-14',
 'Mon-Fri12/5-9',
 'Mon-Fri3/20-24',
 'Fri, Mon-Thu6/9, 12-15']

In [ ]:
import re

# This function cleans the list above and outputs a cleaned list of dates. However, we don't yet have the years, so we will need another helper function for that!
def expand_dates(date_list):
    all_days = []
    
    for entry in date_list:
        # Remove weekday text (everything before first digit)
        entry = re.sub(r'^[^0-9]+', '', entry)
        
        # Split by comma (handles special June final dates of being Friday + Monday-Thurs)
        parts = entry.split(',')
        
        month = None
        
        for part in parts:
            part = part.strip()
            
            # Match month/day-range
            match = re.match(r'(\d+)/(\d+)(?:-(\d+))?', part)
            if match:
                month = match.group(1)
                start = int(match.group(2))
                end = int(match.group(3)) if match.group(3) else start
                
                for day in range(start, end + 1):
                    all_days.append(f"{month}/{day}")
    
    return all_days

expanded = expand_dates(cleanDates)
# print(expanded)

['12/9', '12/10', '12/11', '12/12', '12/13', '3/17', '3/18', '3/19', '3/20', '3/21', '6/6', '12/11', '12/12', '12/13', '12/14', '12/15', '3/18', '3/19', '3/20', '3/21', '3/22', '6/7', '12/5', '12/6', '12/7', '12/8', '12/9', '3/20', '3/21', '3/22', '3/23', '3/24', '6/9', '12/6', '12/7', '12/8', '12/9', '12/10', '3/14', '3/15', '3/16', '3/17', '3/18', '6/3', '12/14', '12/15', '12/16', '12/17', '12/18', '3/15', '3/16', '3/17', '3/18', '3/19', '6/4', '12/9', '12/10', '12/11', '12/12', '12/13', '3/16', '3/17', '3/18', '3/19', '3/20', '6/5', '12/10', '12/11', '12/12', '12/13', '12/14', '3/18', '3/19', '3/20', '3/21', '3/22', '6/7', '12/11', '12/12', '12/13', '12/14', '12/15', '3/19', '3/20', '3/21', '3/22', '3/23', '6/8', '12/5', '12/6', '12/7', '12/8', '12/9', '3/20', '3/21', '3/22', '3/23', '3/24', '6/9']


In [ ]:
# sorted(set(expanded))

['12/10',
 '12/11',
 '12/12',
 '12/13',
 '12/14',
 '12/15',
 '12/16',
 '12/17',
 '12/18',
 '12/5',
 '12/6',
 '12/7',
 '12/8',
 '12/9',
 '3/14',
 '3/15',
 '3/16',
 '3/17',
 '3/18',
 '3/19',
 '3/20',
 '3/21',
 '3/22',
 '3/23',
 '3/24',
 '6/3',
 '6/4',
 '6/5',
 '6/6',
 '6/7',
 '6/8',
 '6/9']

In [ ]:


full_dates = []

current_year = None
current_quarter = None

# Unfortunately, this isn't very pretty but the only solution I found was to iterate through EVERY scraped line
for line in lines:
    
    # Detect quarter header
    header_match = re.search(r'(Fall|Winter|Spring|Summer)\s+(\d{4})', line)
    if header_match:
        current_quarter = header_match.group(1)
        current_year = int(header_match.group(2))
        continue
    
    # Find all MM/DD patterns or ranges
    matches = re.findall(r'(\d+)/(\d+)(?:-(\d+))?', line)
    
    for match in matches:
        month = int(match[0])
        start_day = int(match[1])
        end_day = int(match[2]) if match[2] else start_day
        
        for day in range(start_day, end_day + 1):
            
            year = current_year
            
            # Handle Winter year crossover
            if current_quarter == "Winter" and month == 12:
                year = current_year - 1
            
            full_dates.append(f"{month:02d}/{day:02d}/{year}")

# Remove duplicates if desired  
full_dates = sorted(set(full_dates))
# Unfortunately as a result, this list contains every date mentioned, not just finals. So, we will once again need another helper function to filter out the final dates
# At least it has the years!
# print(full_dates)

['01/01/2017', '01/01/2018', '01/01/2019', '01/01/2020', '01/01/2023', '01/02/2016', '01/02/2022', '01/03/2020', '01/03/2022', '01/03/2025', '01/04/2019', '01/04/2021', '01/05/2018', '01/05/2024', '01/06/2017', '01/06/2020', '01/06/2023', '01/06/2025', '01/07/2019', '01/08/2018', '01/08/2024', '01/09/2017', '01/09/2023', '01/15/2018', '01/15/2024', '01/16/2017', '01/16/2023', '01/17/2022', '01/18/2021', '01/20/2020', '01/20/2025', '01/21/2019', '02/15/2021', '02/17/2020', '02/17/2025', '02/18/2019', '02/19/2018', '02/19/2024', '02/20/2017', '02/20/2023', '02/21/2022', '03/11/2022', '03/12/2021', '03/13/2020', '03/14/2022', '03/14/2025', '03/15/2019', '03/15/2021', '03/15/2022', '03/15/2024', '03/16/2018', '03/16/2020', '03/16/2021', '03/16/2022', '03/17/2017', '03/17/2020', '03/17/2021', '03/17/2022', '03/17/2023', '03/17/2025', '03/18/2019', '03/18/2020', '03/18/2021', '03/18/2022', '03/18/2024', '03/18/2025', '03/19/2018', '03/19/2019', '03/19/2020', '03/19/2021', '03/19/2024', '03/1

In [ ]:
final_days_no_year = [
 '12/10','12/11','12/12','12/13','12/14','12/15','12/16','12/17','12/18',
 '12/5','12/6','12/7','12/8','12/9',
 '3/14','3/15','3/16','3/17','3/18','3/19','3/20','3/21','3/22','3/23','3/24',
 '6/3','6/4','6/5','6/6','6/7','6/8','6/9'
]

# normalize to zero-padded format to match full_dates
final_days_set = set(
    f"{int(m):02d}/{int(d):02d}"
    for m, d in (date.split('/') for date in final_days_no_year)
)

filtered_final_dates = [
    full_date
    for full_date in full_dates
    if full_date[:5] in final_days_set
]

filtered_final_dates = sorted(filtered_final_dates)


# Finally, after comparing final dates from expanded to the full_dates, we can extract just the final dates with years.
print(filtered_final_dates)

['03/14/2022', '03/14/2025', '03/15/2019', '03/15/2021', '03/15/2022', '03/15/2024', '03/16/2018', '03/16/2020', '03/16/2021', '03/16/2022', '03/17/2017', '03/17/2020', '03/17/2021', '03/17/2022', '03/17/2023', '03/17/2025', '03/18/2019', '03/18/2020', '03/18/2021', '03/18/2022', '03/18/2024', '03/18/2025', '03/19/2018', '03/19/2019', '03/19/2020', '03/19/2021', '03/19/2024', '03/19/2025', '03/20/2017', '03/20/2018', '03/20/2019', '03/20/2020', '03/20/2023', '03/20/2024', '03/20/2025', '03/21/2017', '03/21/2018', '03/21/2019', '03/21/2023', '03/21/2024', '03/21/2025', '03/22/2017', '03/22/2018', '03/22/2019', '03/22/2023', '03/22/2024', '03/23/2017', '03/23/2018', '03/23/2023', '03/24/2017', '03/24/2022', '03/24/2023', '06/03/2021', '06/03/2022', '06/04/2020', '06/04/2021', '06/05/2020', '06/05/2025', '06/06/2019', '06/06/2024', '06/06/2025', '06/07/2018', '06/07/2019', '06/07/2024', '06/08/2017', '06/08/2018', '06/08/2023', '06/09/2017', '06/09/2022', '06/09/2023', '12/05/2016', '12/0